# 99 · Fusión de Grafos Multidimensional

Fusiona los tres grafos del pipeline en un único grafo enriquecido:
- **Dimensiones**: transcripción, año, vestimenta

> Para añadir dimensiones nuevas: genera `output/DIM/grafo_DIM.gexf` y añade la entrada a `GEXF_SOURCES`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import networkx as nx
import pandas as pd

BASE = '/content/drive/MyDrive/TFM-Sara'

# Registro de dimensiones — añade entradas aquí para nuevas dimensiones
GEXF_SOURCES = {
    'transcripcion': f'{BASE}/output/transcripciones/knowledge_graph_palabras.gexf',
    'anyo':          f'{BASE}/output/predicciones_año/grafo_años.gexf',
    'vestimenta':    f'{BASE}/output/vestimenta/grafo_vestimenta.gexf',
}
SUPPORT_CSV = f'{BASE}/support_data/input.csv'
OUTPUT_DIR  = f'{BASE}/output/grafo_final'
OUTPUT_GEXF = f'{OUTPUT_DIR}/grafo_multidimensional.gexf'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'NetworkX {nx.__version__}  |  Rutas OK')

NetworkX 3.6.1  |  Rutas OK


In [3]:
# Cargar metadatos de soporte (filename → fila CSV)
df_meta = pd.read_csv(SUPPORT_CSV, dtype=str).fillna('')
meta = {row['filename']: row.to_dict() for _, row in df_meta.iterrows()}
print(f'Metadatos cargados: {len(meta)} registros')
print(f'Columnas: {list(df_meta.columns)}')

Metadatos cargados: 312 registros
Columnas: ['nom_arxiu', 'nom_fons', 'caption', 'toponims', 'noms_propis', 'url', 'filename', 'year']


In [4]:
# Helper tolerante para leer GEXF: promueve a `double` cualquier atributo
# declarado como `integer`/`long` cuyos valores contengan decimales (caso
# `size = 13.333...` en el grafo de transcripciones), evitando el ValueError
# de NetworkX al castear a int.
import io
import re
import xml.etree.ElementTree as ET
import networkx as nx


def _read_gexf_tolerant(path):
    """Lee un GEXF promoviendo a 'double' cualquier atributo declarado
    integer/long cuyos valores reales contengan decimales.
    Evita el ValueError de NetworkX al castear strings como
    '13.333333333333332' a int.
    """
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()

    ns_match = re.search(r'xmlns="([^"]+)"', text)
    NS = ns_match.group(1) if ns_match else ''
    NSP = '{' + NS + '}' if NS else ''
    if NS:
        ET.register_namespace('', NS)

    root = ET.fromstring(text)

    vals_by_id: dict[str, list[str]] = {}
    for av in root.iter(NSP + 'attvalue'):
        vals_by_id.setdefault(av.get('for'), []).append(av.get('value', ''))

    fixes = []
    for attr in root.iter(NSP + 'attribute'):
        if attr.get('type') in ('integer', 'long'):
            aid = attr.get('id')
            for v in vals_by_id.get(aid, []):
                if not v:
                    continue
                try:
                    int(v)
                except ValueError:
                    attr.set('type', 'double')
                    fixes.append((aid, attr.get('title')))
                    break

    if fixes:
        print(f'  promovidos a double: {fixes}')

    buf = io.BytesIO()
    ET.ElementTree(root).write(buf, encoding='utf-8', xml_declaration=True)
    buf.seek(0)
    return nx.read_gexf(buf)


print('Helper _read_gexf_tolerant listo')

Helper _read_gexf_tolerant listo


In [5]:
# Fusión de grafos por dimensión
# - Cada GEXF de origen reutiliza ids de arista (0,1,2...) y, al combinarlos,
#   Gephi rechaza el resultado con
#   `Graph.mergeUndirectedEdgeWithKey: inconsistency detected`.
#   Por eso descartamos el atributo `id` de nodos y aristas: dejamos que
#   `nx.write_gexf` regenere ids únicos.
# - Nodos imagen comparten id (filename) entre dimensiones; nodos hub
#   (año, vestimenta, palabra) conservan sus ids propios.
G_final = nx.Graph()

def _clean(attrs: dict) -> dict:
    return {k: v for k, v in attrs.items() if k != 'id'}

for dim_name, gexf_path in GEXF_SOURCES.items():
    if not os.path.exists(gexf_path):
        print(f'\u26a0\ufe0f  Skipping {dim_name}: no encontrado en {gexf_path}')
        continue

    G_dim = _read_gexf_tolerant(gexf_path)
    print(f'\u2705 {dim_name}: {G_dim.number_of_nodes()} nodos \u00b7 {G_dim.number_of_edges()} aristas')

    for node_id, attrs in G_dim.nodes(data=True):
        attrs = _clean(attrs)
        if G_final.has_node(node_id):
            for k, v in attrs.items():
                if k not in G_final.nodes[node_id]:
                    G_final.nodes[node_id][k] = v
        else:
            G_final.add_node(node_id, **attrs)

    for u, v, attrs in G_dim.edges(data=True):
        attrs = _clean(attrs)
        if not G_final.has_edge(u, v):
            G_final.add_edge(u, v, **attrs)
        else:
            existing_w = float(G_final[u][v].get('weight', 1.0))
            new_w = float(attrs.get('weight', 1.0))
            if new_w > existing_w:
                G_final[u][v]['weight'] = new_w

print(f'\n\U0001f517 Grafo fusionado: {G_final.number_of_nodes()} nodos \u00b7 {G_final.number_of_edges()} aristas')

  promovidos a double: [('4', 'size')]
✅ transcripcion: 224 nodos · 156 aristas
✅ anyo: 378 nodos · 1957 aristas
✅ vestimenta: 521 nodos · 2305 aristas

🔗 Grafo fusionado: 746 nodos · 4418 aristas


In [6]:
# Enriquecer nodos imagen con metadatos del CSV de soporte
enriquecidos = 0
for node_id, attrs in G_final.nodes(data=True):
    if attrs.get('dimension') == 'imagen' and node_id in meta:
        row = meta[node_id]
        G_final.nodes[node_id].update({
            'nom_arxiu':   row.get('nom_arxiu', ''),
            'nom_fons':    row.get('nom_fons', ''),
            'caption':     row.get('caption', ''),
            'toponims':    row.get('toponims', ''),
            'noms_propis': row.get('noms_propis', ''),
            'url':         row.get('url', ''),
            'year_csv':    row.get('year', ''),
        })
        enriquecidos += 1

sin_meta = sum(
    1 for _, d in G_final.nodes(data=True)
    if d.get('dimension') == 'imagen' and d.get('nom_arxiu', '') == ''
)
print(f'Nodos imagen enriquecidos : {enriquecidos}')
print(f'Nodos imagen sin metadatos: {sin_meta}')

Nodos imagen enriquecidos : 0
Nodos imagen sin metadatos: 569


In [7]:
# Post-procesado: TF-IDF + filtros + comunidades para grafo legible en Gephi
# Ajusta los parámetros y re-ejecuta esta celda + save_output.

MIN_WEIGHT         = 0.0    # >0 elimina aristas débiles (prueba 0.3 / 0.5)
MIN_CONFIDENCE     = 0.0    # >0 filtra nodos año poco fiables (prueba 0.5)
MAX_PHOTO_FREQ     = 0.5    # prendas en > X% de fotos se eliminan (1.0 = off)
REMOVE_ISOLATED    = True   # quitar nodos sin conexiones tras filtrar
DETECT_COMMUNITIES = True   # añadir atributo `community` (Louvain)

import math, colorsys

G = G_final.copy()
n_photos = max(1, sum(1 for _, d in G.nodes(data=True) if d.get('dimension') == 'imagen'))

# 1) TF-IDF en aristas lleva_puesto (prendas raras pesan más, comunes menos)
prenda_df = {}
for u, v, d in G.edges(data=True):
    if d.get('relation') == 'lleva_puesto':
        prenda = u if G.nodes[u].get('dimension') == 'vestimenta' else v
        prenda_df[prenda] = prenda_df.get(prenda, 0) + 1

for u, v, d in G.edges(data=True):
    if d.get('relation') == 'lleva_puesto':
        prenda = u if G.nodes[u].get('dimension') == 'vestimenta' else v
        df = prenda_df.get(prenda, 1)
        idf = math.log((n_photos + 1) / (df + 1)) + 1.0
        d['weight'] = round(float(d.get('weight', 1.0)) * idf, 4)

# 2) Prendas demasiado comunes fuera
if MAX_PHOTO_FREQ < 1.0:
    common = [p for p, df in prenda_df.items() if df / n_photos > MAX_PHOTO_FREQ]
    G.remove_nodes_from(common)
    print(f'Prendas eliminadas (>{int(MAX_PHOTO_FREQ*100)}% fotos): {len(common)} -> {common}')

# 3) Nodos año por confianza
if MIN_CONFIDENCE > 0:
    drop = [n for n, d in G.nodes(data=True)
            if d.get('dimension') in ('año', 'anyo')
            and float(d.get('confianza', d.get('confidence', 1.0)) or 1.0) < MIN_CONFIDENCE]
    G.remove_nodes_from(drop)
    print(f'Nodos año con confianza < {MIN_CONFIDENCE}: {len(drop)} eliminados')

# 4) Aristas débiles
if MIN_WEIGHT > 0:
    weak = [(u, v) for u, v, d in G.edges(data=True) if float(d.get('weight', 1.0)) < MIN_WEIGHT]
    G.remove_edges_from(weak)
    print(f'Aristas eliminadas (weight < {MIN_WEIGHT}): {len(weak)}')

# 5) Nodos huérfanos
if REMOVE_ISOLATED:
    iso = list(nx.isolates(G))
    G.remove_nodes_from(iso)
    print(f'Nodos huérfanos eliminados: {len(iso)}')

# 6) Comunidades Louvain -> atributo `community`
n_comms = 1
if DETECT_COMMUNITIES and G.number_of_edges() > 0:
    comms = nx.community.louvain_communities(G, weight='weight', seed=42)
    for cid, members in enumerate(comms):
        for n in members:
            G.nodes[n]['community'] = cid
    n_comms = len(comms)
    print(f'Comunidades detectadas: {n_comms}')

# 7) viz: color por comunidad + tamaño por grado (Gephi lo lee directo)
def _rgb(i, total):
    r, g, b = colorsys.hls_to_rgb((i / max(total, 1)) % 1.0, 0.55, 0.65)
    return {'r': int(r*255), 'g': int(g*255), 'b': int(b*255), 'a': 1.0}

palette = [_rgb(i, n_comms) for i in range(max(n_comms, 1))]
for n, d in G.nodes(data=True):
    cid = int(d.get('community', 0)) if d.get('community') is not None else 0
    deg = G.degree(n)
    d['degree_static'] = deg
    d['viz'] = {'color': palette[cid % len(palette)],
                'size': float(6 + math.log1p(deg) * 5)}

G_final = G
print(f'\n📊 Tras post-procesado: {G_final.number_of_nodes()} nodos · {G_final.number_of_edges()} aristas')


Prendas eliminadas (>50% fotos): 1 -> ['necktie']
Nodos huérfanos eliminados: 5
Comunidades detectadas: 12

📊 Tras post-procesado: 740 nodos · 4120 aristas


In [8]:
# Guardar grafo multidimensional + resumen de estadísticas
nx.write_gexf(G_final, OUTPUT_GEXF)

dims: dict[str, int] = {}
for _, attrs in G_final.nodes(data=True):
    d = attrs.get('dimension', 'unknown')
    dims[d] = dims.get(d, 0) + 1

print(f'\u2705 Guardado en: {OUTPUT_GEXF}')
print(f'   Nodos totales : {G_final.number_of_nodes()}')
print(f'   Aristas totales: {G_final.number_of_edges()}')
for d, n in sorted(dims.items()):
    print(f'   \u00b7 {d}: {n} nodos')

✅ Guardado en: /content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf
   Nodos totales : 740
   Aristas totales: 4120
   · año: 17 nodos
   · imagen: 564 nodos
   · transcripcion: 127 nodos
   · vestimenta: 32 nodos
